# Ingesta SECOP II > Procesos

Se descargan los procesos de contratación de SECOP II para el año y mes seleccionados. 


## 1. Configuración

Destino: `procesos_filas_api`. Si la API no permite descargar todos los registros, se guardan los disponibles y se informa que la descarga está incompleta.


In [1]:
import os
import sys
from datetime import date
from pathlib import Path

from pymongo import MongoClient

NB_DIR = Path("/home/jovyan/work/spark-apps/notebooks")
if not NB_DIR.exists():
    NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR.parent / "src" / "ingestion"))

from secop_to_mongo import ingestar_periodo

FUENTE = "procesos"
NOMBRE = "SECOP II Procesos de Contratación"
DATASET_ID = "p6dx-8zbt"
DATE_FIELD = "fecha_de_publicacion_del"
PK_NEGOCIO = "id_del_proceso"
COLECCION = "procesos_filas_api"

# --- parámetros (año y mes configurables; fuente fija) ---
ANIO = 2026
MES = 9
REEJECUTAR = False  # True solo para comprobar idempotencia (vuelve a descargar)

MONGO_DB = "secop"
PROTEGER = ("procesos_raw", "contratos_raw")
PK_FILA = "socrata_id"
BATCH_SIZE = 5000
SLEEP_SECONDS = 1

inicio = date(ANIO, MES, 1)
fin = date(ANIO + 1, 1, 1) if MES == 12 else date(ANIO, MES + 1, 1)
START_DATE = inicio.isoformat()
END_DATE = fin.isoformat()
PERIODO = f"{ANIO:04d}-{MES:02d}"

print("Fuente:", FUENTE, "|", NOMBRE)
print("Dataset:", DATASET_ID)
print("Colección:", COLECCION, "| PK fila:", PK_FILA)
print("Fecha:", DATE_FIELD, "| PK negocio:", PK_NEGOCIO)
print("Periodo:", START_DATE, "<= fecha <", END_DATE)
print("Colecciones protegidas:", ", ".join(PROTEGER))


Fuente: procesos | SECOP II Procesos de Contratación
Dataset: p6dx-8zbt
Colección: procesos_filas_api | PK fila: socrata_id
Fecha: fecha_de_publicacion_del | PK negocio: id_del_proceso
Periodo: 2026-09-01 <= fecha < 2026-10-01
Colecciones protegidas: procesos_raw, contratos_raw


## 2. Conexión

La URI sale de `MONGO_URI`. Las colecciones históricas no son destino.


In [2]:
uri = os.environ.get("MONGO_URI")
if not uri:
    raise RuntimeError("Falta MONGO_URI en el entorno del contenedor.")
if COLECCION in PROTEGER:
    raise RuntimeError(f"La colección de destino no puede ser {COLECCION}.")

client = MongoClient(uri, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
db = client[MONGO_DB]

conteo_antes = {
    col: db[col].estimated_document_count() if col in db.list_collection_names() else 0
    for col in PROTEGER
}

print("MongoDB: OK")
print("Colecciones:", ", ".join(sorted(db.list_collection_names())) or "(ninguna)")
for col, n in conteo_antes.items():
    print(f"{col} (no se escribe): {n:,}".replace(",", "."))
print("App Token presente:", bool(os.environ.get("SOCRATA_APP_TOKEN")))


RuntimeError: Falta MONGO_URI en el entorno del contenedor.

## 3. Validación y descarga

Se descargan los registros en grupos de 5.000 y se almacenan en MongoDB, evitando duplicados mediante su identificador.

In [ ]:
kwargs = dict(
    db=db,
    fuente=FUENTE,
    dataset_id=DATASET_ID,
    date_field=DATE_FIELD,
    coleccion=COLECCION,
    pk_negocio=PK_NEGOCIO,
    start_date=START_DATE,
    end_date=END_DATE,
    periodo=PERIODO,
    nombre=NOMBRE,
    pk_fila=PK_FILA,
    batch_size=BATCH_SIZE,
    sleep_seconds=SLEEP_SECONDS,
    proteger=PROTEGER,
    resume=False,
)

resultado = ingestar_periodo(**kwargs)
n_primera = resultado.get("almacenadas", 0)

if REEJECUTAR and resultado.get("estado") in ("COMPLETA", "PARCIAL"):
    print("\n--- Reejecución (idempotencia) ---")
    otra = ingestar_periodo(**kwargs)
    n_segunda = otra.get("almacenadas", 0)
    print("Almacenadas 1.ª:", f"{n_primera:,}".replace(",", "."),
          "| 2.ª:", f"{n_segunda:,}".replace(",", "."))
    if n_segunda != n_primera:
        raise RuntimeError("La reejecución cambió el número de documentos.")
    print("Idempotencia: OK (mismo recuento).")
    resultado = otra

conteo_despues = {
    col: db[col].estimated_document_count() if col in db.list_collection_names() else 0
    for col in PROTEGER
}
for col in PROTEGER:
    if conteo_despues[col] != conteo_antes[col]:
        raise RuntimeError(f"Se modificó {col}: {conteo_antes[col]} -> {conteo_despues[col]}")
    print(f"{col} intacta:", f"{conteo_despues[col]:,}".replace(",", "."))


procesos | SECOP II Procesos de Contratación | p6dx-8zbt
count API del mes: 158.830
count API sin filtro: 9.377.332
Muestra publicada limitada: False
muestra mes: 3 | PK negocio ok: True | fecha ok: True
Fila mes en offset 1000: True
Identificador de fila: Se usa :id de Socrata (socrata_id).
  :id disponible: True | único: True | estable: True | sin solape: True
[INFO] Iniciando ingesta (dataset=p6dx-8zbt, campo_fecha=fecha_de_publicacion_del, start_date=2026-09-01, end_date=2026-10-01) desde offset=0
[OK] Upsert lote=5000 acumulado=5000 offset=5000
[OK] Upsert lote=5000 acumulado=10000 offset=10000
[OK] Upsert lote=5000 acumulado=15000 offset=15000
[OK] Upsert lote=5000 acumulado=20000 offset=20000
[OK] Upsert lote=5000 acumulado=25000 offset=25000
[OK] Upsert lote=5000 acumulado=30000 offset=30000
[OK] Upsert lote=5000 acumulado=35000 offset=35000
[OK] Upsert lote=5000 acumulado=40000 offset=40000
[OK] Upsert lote=5000 acumulado=45000 offset=45000
[OK] Upsert lote=5000 acumulado=5000

## 4. Resumen

Se verifica que la cantidad de registros almacenados coincida con la cantidad reportada por la API


In [4]:
print("Resumen de la ingesta")
print("--------------------")
print(f"Fuente: {resultado['fuente']}")
print(f"Dataset: {resultado['dataset_id']}")
print(f"Colección: {resultado['coleccion']}")
print(f"Registros en la API: {resultado['n_api']:,}")
print(f"Registros descargados: {resultado['recuperadas']:,}")
print(f"Registros almacenados: {resultado['almacenadas']:,}")
print(f"Procesos diferentes: {resultado.get('n_negocio', 0):,}")
print(f"Identificador de fila: {resultado['id_usado']}")
print(f"Estado: {resultado['estado']}")
print(f"Observación: {resultado['motivo']}")


Resumen de la ingesta
--------------------
Fuente: procesos
Dataset: p6dx-8zbt
Colección: procesos_filas_api
Registros en la API: 158.830
Registros descargados: 158.830
Registros almacenados: 158.830
Procesos diferentes: 146.947
Identificador de fila: socrata_id (:id)
Estado: COMPLETA
Observación: Filas recuperadas = almacenadas = count(*) del mes.
